Use kernel `Python 3.11.4` for this script to avoid SpaGCN version conflict.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import scanpy as sc
import SpaGCN as spg
import torch

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [ ]:
# Settings
ROI_dict = {"Isocortex": {"MERSCOPE_WT_1": [1, 3, 15, 12, 13, 23], "MERSCOPE_AD_1": [2, 5, 11, 17, 26]},
            "HPF": {"MERSCOPE_WT_1": [101, 102, 103], "MERSCOPE_AD_1": [101, 102, 103]}}
ROI = "HPF"

plot_size_dict = {
    "Isocortex": {"MERSCOPE_WT_1": 15, "MERSCOPE_AD_1": 18},
    "HPF": {"MERSCOPE_WT_1": 20, "MERSCOPE_AD_1": 30}
}

plot_cutoff_dict = {
    "Isocortex": {"MERSCOPE_WT_1": {"y": 2500, "xmin": 0, "xmax": 5000, "a": 0.4, "b": 1100}, "MERSCOPE_AD_1": {"y": 3500, "xmin": 8000, "xmax": 12500, "a": 1.75, "b": -16680}},
    "HPF": {"MERSCOPE_WT_1": {"y": 2500, "xmin": 0, "xmax": 5000, "a": 0, "b": 0}, "MERSCOPE_AD_1": {"y": 2500, "xmin": 0, "xmax": 5000, "a": 0, "b": 0}}
}

num_cluster_dict = {
    "Isocortex": {"MERSCOPE_WT_1": 12, "MERSCOPE_AD_1": 18},
    "HPF": {"MERSCOPE_WT_1": 8, "MERSCOPE_AD_1": 10}
}

mapping_dict = {
    "Isocortex": {
    "MERSCOPE_WT_1": {0: "L5", 1: "L6", 2: "L1", 3: "L2/3", 4: "Others", 5: "L4", 6: "RSP", 7: "L1", 8: "RSP", 9: "L2/3", 11: "Others", 12: "Others"},
    "MERSCOPE_AD_1": {0: "L5", 1: "L1", 2: "L2/3", 3: "L4", 4: "RSP", 5: "L5", 6: "L6", 7: "RSP", 8: "Others", 9: "L6", 10: "RSP", 11: "Others", 13: "L1", 14: "Others", 15: "L2/3", 17: "Others", 18: "Others"}
    },
    "HPF": {
    # "MERSCOPE_WT_1": {0: "Neuropil", 1: "CA3", 2: "Neuropil", 3: "CA1", 4: "Neuropil", 5 : "Neuropil", 6: "DG", 7: "DG"},
    # "MERSCOPE_AD_1": {0: "CA1", 1: "Neuropil", 2: "DG", 3: "CA3", 4: "DG", 5: "Neuropil", 6: "Neuropil", 7: "Neuropil", 8: "CA1", 9: "CA1"}
    "MERSCOPE_WT_1": {0: "Neuropil", 1: "CA3", 2: "Neuropil", 3: "CA1", 4: "Neuropil", 5 : "Neuropil", 6: "Neuropil", 7: "DG"},
    "MERSCOPE_AD_1": {0: "Neuropil", 1: "Neuropil", 2: "Neuropil", 3: "CA3", 4: "DG", 5: "CA1", 6: "CA1", 7: "Neuropil", 8: "Neuropil", 9: "Neuropil"}
    }
}

keep_region_markers = False

In [ ]:
# File paths
comparison_path = f"../output/MERSCOPE_WT_AD_comparison/"

# Read spots
spots = sc.read_h5ad(comparison_path + "neuropil_subdomains_spots.h5ad")
spots = spots[spots.obs["brain_area"] != "Unknown"].copy()
spots_full = spots.copy()
spots_full.obs["layer_labels"] = None

# Keep region markers
if keep_region_markers:
    gene_panel = pd.read_csv("../data/MERSCOPE_WT_1/processed_data/gene_panel.csv")
    markers = list(gene_panel[(gene_panel["Region markers"] != " ") | (gene_panel["Cell type markers"] != " ")]["Gene"])
    markers = [i for i in markers if i in spots.var_names]
    spots = spots_full[:, markers].copy()

# Split into WT and AD
wt_mask = spots.obs["batch"] == "MERSCOPE_WT_1"
ad_mask = spots.obs["batch"] == "MERSCOPE_AD_1"

### Annotate WT

In [ ]:
# Select ROI
spots_WT = spots[wt_mask].copy()
spots_WT = spots_WT[spots_WT.obs["region_labels"].isin(ROI_dict[ROI]["MERSCOPE_WT_1"])].copy()

sc.set_figure_params(scanpy = True, figsize = (6, 10))
ax = sc.pl.scatter(spots_WT, alpha = 1, x = "global_x", y = "global_y", color = "region_labels", size = plot_size_dict[ROI]["MERSCOPE_WT_1"], title = " ", show = False)
ax.grid(False)
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
# Spot coordinates
x_array = spots_WT.obs["global_x"].tolist()
y_array = spots_WT.obs["global_y"].tolist()

In [ ]:
# Adjacency matrix
s = 1
b = 49
adj = spg.calculate_adj_matrix(x = x_array, y = y_array, histology = False)

In [ ]:
# Pre-processing
spots_WT.var_names_make_unique()
spg.prefilter_genes(spots_WT, min_cells = 3)
spg.prefilter_specialgenes(spots_WT)
sc.pp.normalize_total(spots_WT, target_sum = 1e4)
sc.pp.log1p(spots_WT)

In [ ]:
# Set hyperparameters
p = 0.5
l = spg.search_l(p, adj, start = 0.01, end = 1000, tol = 0.01, max_run = 100)

n_clusters = num_cluster_dict[ROI]["MERSCOPE_WT_1"]
r_seed = t_seed = n_seed = 1
res = spg.search_res(spots_WT, adj, l, n_clusters, start = 0.7, step = 0.1, tol = 5e-3, lr = 0.05, max_epochs = 20, r_seed = r_seed, t_seed = t_seed, n_seed = n_seed)

In [ ]:
# Run SpaGCN
clf = spg.SpaGCN()
clf.set_l(l)

random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)

clf.train(spots_WT, adj, init_spa = True, init = "louvain", res = res, tol = 5e-3, lr = 0.05, max_epochs = 200)
y_pred, prob = clf.predict()
spots_WT.obs["pred"] = y_pred
spots_WT.obs["pred"] = spots_WT.obs["pred"].astype("category")

adj_2d = spg.calculate_adj_matrix(x = x_array, y = y_array, histology = False)
refined_pred = spg.refine(sample_id = spots_WT.obs.index.tolist(), pred = spots_WT.obs["pred"].tolist(), dis = adj_2d, shape = "square")
spots_WT.obs["refined_pred"] = refined_pred
spots_WT.obs["refined_pred"] = spots_WT.obs["refined_pred"].astype("category")

In [ ]:
# Plot cluster labels
sc.set_figure_params(scanpy = True, figsize = (6, 10))
ax = sc.pl.scatter(spots_WT, alpha = 1, x = "global_x", y = "global_y", color = "refined_pred", size = plot_size_dict[ROI]["MERSCOPE_WT_1"], title = " ", show = False)
ax.grid(False)
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
# Map cluster labels
spots_WT.obs["layer_labels"] = spots_WT.obs["refined_pred"].map(mapping_dict[ROI]["MERSCOPE_WT_1"])

In [ ]:
# Plot layer labels
if ROI == "Isocortex":
    a = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["a"]
    b = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["b"]
    x_vals = np.linspace(3200, 5200, 200)
    y_vals = a * x_vals + b

    ax.plot(x_vals, y_vals, color="blue", linestyle="--", linewidth=2)
    sc.set_figure_params(scanpy = True, figsize = (6, 10))
    ax = sc.pl.scatter(spots_WT, alpha = 1, x = "global_x", y = "global_y", color = "layer_labels", size = plot_size_dict[ROI]["MERSCOPE_WT_1"], title = " ", show = False)
    ax.grid(False)
    ax.hlines(y = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["y"], xmin = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["xmin"], xmax = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["xmax"], color = "red", linestyle = "--")
    ax.plot(x_vals, y_vals, color="blue", linestyle="--")
    ax.set_aspect("equal", "box")
    plt.show()

In [ ]:
# Refine layer labels
if ROI == "Isocortex":
    disgard_mask_1 = (spots_WT.obs["global_y"] < plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["y"])
    disgard_mask_2 = spots_WT.obs["global_y"] < (a * spots_WT.obs["global_x"] + b)
    disgard_mask_3 = ((spots_WT.obs["layer_labels"] == "RSP") & (spots_WT.obs["global_x"] > 1500))
    disgard_mask_4 = ((spots_WT.obs["layer_labels"] == "L1") & (spots_WT.obs["global_x"] > 800) & (spots_WT.obs["global_x"] < 3000) & (spots_WT.obs["global_y"] < 5000))
    disgard_mask_5 = ((spots_WT.obs["layer_labels"] == "L6") & (spots_WT.obs["global_x"] > 2000) & (spots_WT.obs["global_x"] < 3000) & (spots_WT.obs["global_y"] < 4000))
    disgard_mask = (disgard_mask_1) | (disgard_mask_2) | (disgard_mask_3) | (disgard_mask_4) | (disgard_mask_5)
    spots_WT.obs.loc[disgard_mask, "layer_labels"] = "Others"

In [ ]:
# Plot refined layer labels
sc.set_figure_params(scanpy = True, figsize = (6, 10))
ax = sc.pl.scatter(spots_WT, alpha = 1, x = "global_x", y = "global_y", color = "layer_labels", size = plot_size_dict[ROI]["MERSCOPE_WT_1"], title = " ", show = False)
ax.grid(False)
if ROI == "Isocortex":
    ax.hlines(y = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["y"], xmin = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["xmin"], xmax = plot_cutoff_dict[ROI]["MERSCOPE_WT_1"]["xmax"], color = "red", linestyle = "--")
    ax.plot(x_vals, y_vals, color="blue", linestyle="--")
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
# Save refined layer labels
spots_full.obs.loc[wt_mask, "layer_labels"] = spots_WT.obs["layer_labels"].copy()

### Annotate AD

In [ ]:
# Select ROI
spots_AD = spots[ad_mask].copy()
spots_AD = spots_AD[spots_AD.obs["region_labels"].isin(ROI_dict[ROI]["MERSCOPE_AD_1"])].copy()

sc.set_figure_params(scanpy = True, figsize = (6, 10))
ax = sc.pl.scatter(spots_AD, alpha = 1, x = "global_x", y = "global_y", color = "region_labels", size = plot_size_dict[ROI]["MERSCOPE_AD_1"], title = " ", show = False)
ax.grid(False)
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
# Spot coordinates
x_array = spots_AD.obs["global_x"].tolist()
y_array = spots_AD.obs["global_y"].tolist()

In [ ]:
# Adjacency matrix
s = 1
b = 49
adj = spg.calculate_adj_matrix(x = x_array, y = y_array, histology = False)

In [ ]:
# Pre-processing
spots_AD.var_names_make_unique()
spg.prefilter_genes(spots_AD, min_cells = 3)
spg.prefilter_specialgenes(spots_AD)
sc.pp.normalize_total(spots_AD, target_sum = 1e4)
sc.pp.log1p(spots_AD)

In [ ]:
# Set hyperparameters
p = 0.5
l = spg.search_l(p, adj, start = 0.01, end = 1000, tol = 0.01, max_run = 100)

n_clusters = num_cluster_dict[ROI]["MERSCOPE_AD_1"]
r_seed = t_seed = n_seed = 1
res = spg.search_res(spots_AD, adj, l, n_clusters, start = 0.7, step = 0.1, tol = 5e-3, lr = 0.05, max_epochs = 20, r_seed = r_seed, t_seed = t_seed, n_seed = n_seed)

In [ ]:
# Run SpaGCN
clf = spg.SpaGCN()
clf.set_l(l)

random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)

clf.train(spots_AD, adj, init_spa = True, init = "louvain", res = res, tol = 5e-3, lr = 0.05, max_epochs = 200)
y_pred, prob = clf.predict()
spots_AD.obs["pred"] = y_pred
spots_AD.obs["pred"] = spots_AD.obs["pred"].astype("category")

adj_2d = spg.calculate_adj_matrix(x = x_array, y = y_array, histology = False)
refined_pred = spg.refine(sample_id = spots_AD.obs.index.tolist(), pred = spots_AD.obs["pred"].tolist(), dis = adj_2d, shape = "square")
spots_AD.obs["refined_pred"] = refined_pred
spots_AD.obs["refined_pred"] = spots_AD.obs["refined_pred"].astype("category")

In [ ]:
# Plot cluster labels
sc.set_figure_params(scanpy = True, figsize = (6, 10))
ax = sc.pl.scatter(spots_AD, alpha = 1, x = "global_x", y = "global_y", color = "refined_pred", size = plot_size_dict[ROI]["MERSCOPE_AD_1"], title = " ", show = False)
ax.grid(False)
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
# Map cluster labels
spots_AD.obs["layer_labels"] = spots_AD.obs["refined_pred"].map(mapping_dict[ROI]["MERSCOPE_AD_1"])

In [ ]:
# Plot layer labels
if ROI == "Isocortex":
    a = 1.75
    b = -16680
    x_vals = np.linspace(11000, 12500, 200)
    y_vals = a * x_vals + b

    ax.plot(x_vals, y_vals, color="blue", linestyle="--", linewidth=2)
    sc.set_figure_params(scanpy = True, figsize = (6, 10))
    ax = sc.pl.scatter(spots_AD, alpha = 1, x = "global_x", y = "global_y", color = "layer_labels", size = plot_size_dict[ROI]["MERSCOPE_AD_1"], title = " ", show = False)
    ax.grid(False)
    ax.hlines(y = plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["y"], xmin = plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["xmin"], xmax = plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["xmax"], color = "red", linestyle = "--")
    ax.plot(x_vals, y_vals, color="blue", linestyle="--")
    ax.set_aspect("equal", "box")
    plt.show()

In [ ]:
# Refine layer labels
if ROI == "Isocortex":
    
    disgard_mask_1 = (spots_AD.obs["global_y"] < plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["y"])
    disgard_mask_2 = spots_AD.obs["global_y"] < (a * spots_AD.obs["global_x"] + b)
    disgard_mask_3 = ((spots_AD.obs["layer_labels"] == "RSP") & (spots_AD.obs["global_x"] > 9200))
    disgard_mask = (disgard_mask_1) | (disgard_mask_2) | (disgard_mask_3)
    spots_AD.obs.loc[disgard_mask, "layer_labels"] = "Others"
    
    change_mask_1 = (spots_AD.obs["layer_labels"] == "L2/3") & (spots_AD.obs["global_x"] < 8500)
    change_mask_2 = (spots_AD.obs["layer_labels"] == "L5") & (spots_AD.obs["global_x"] < 8500)
    change_mask_3 = (spots_AD.obs["layer_labels"] == "L5") & (spots_AD.obs["global_x"] < 8775)
    spots_AD.obs.loc[change_mask_1, "layer_labels"] = "L1"
    spots_AD.obs.loc[change_mask_2, "layer_labels"] = "L1"
    spots_AD.obs.loc[change_mask_3, "layer_labels"] = "RSP"

In [ ]:
# Plot refined layer labels
sc.set_figure_params(scanpy = True, figsize = (6, 10))
ax = sc.pl.scatter(spots_AD, alpha = 1, x = "global_x", y = "global_y", color = "layer_labels", size = plot_size_dict[ROI]["MERSCOPE_AD_1"], title = " ", show = False)
ax.grid(False)
if ROI == "Isocortex":
    ax.hlines(y = plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["y"], xmin = plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["xmin"], xmax = plot_cutoff_dict[ROI]["MERSCOPE_AD_1"]["xmax"], color = "red", linestyle = "--")
    ax.vlines(x = 8775, ymin = 1000, ymax = 6000, color = "red", linestyle = "--")
    ax.plot(x_vals, y_vals, color="blue", linestyle="--")
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
# Save refined layer labels
spots_full.obs.loc[ad_mask, "layer_labels"] = spots_AD.obs["layer_labels"].copy()

### Save data

In [ ]:
spots_save = spots_full[spots_full.obs["layer_labels"].notna()].copy()
spots_save.write_h5ad(comparison_path + f"neuropil_subdomains_spots_{ROI}.h5ad")

In [ ]:
sc.set_figure_params(scanpy = True, figsize = (10, 8))
ax = sc.pl.scatter(spots_save, alpha = 1, x = "global_x", y = "global_y", color = "layer_labels", size = 10, title = " ", show = False)
ax.grid(False)
ax.set_aspect("equal", "box")
plt.show()

In [ ]:
if ROI == "HPF":
    spots_save.obs.to_csv(comparison_path + f"{ROI}_annotation.csv", index = False)